# PS Features

Este notebook construye todas las features necesarias para el cálculo del propensity score y las variables de resultado. Genera la cuadrícula espacial, asigna el tratamiento (presencia de cámara), calcula features fijas de las vialidades, incorpora información de afluencia del metro, y construye tendencias de accidentes previas al tratamiento. También genera las variables de resultado (outcome) mensuales para cada grid.

**Inputs necesarios:**
- `vialidades.json`: Geometría de las vialidades de la Ciudad de México
- `fotocivicas-ubicacion-puntos/fotocivicas-ubicacion-puntos.shp`: Ubicación de las cámaras de velocidad
- `metro-station-coordinates.parquet`: Coordenadas geográficas de las estaciones del metro
- `afluencia-metro-semanal.parquet`: Afluencia semanal del metro por estación
- `classified-incidents.parquet`: Incidentes viales clasificados por nivel de severidad
- `volumen-total-mensual.parquet`: Volumen total de tránsito mensual en la ciudad

**Outputs generados:**
- `grid-propensity-score-features.parquet`: Features para el cálculo del propensity score (tratamiento, atributos de vialidades, afluencia del metro, tendencias de accidentes)
- `grid.parquet`: Cuadrícula espacial con las unidades de análisis
- `outcome-variables.parquet`: Variables de resultado mensuales (accidentes totales, min, pic, fcs) por grid y timestamp

In [1]:
import os
import sys
sys.path.insert(1, '../')

import warnings
import pandas as pd
import geopandas as gpd
from utils import build_grid
from shapely.geometry import Polygon

warnings.filterwarnings("ignore")
PATH_DATA = '../../data/'
GRID_SIZE = 500

In [2]:
INICIO_OPERACIONES = pd.to_datetime("2019-04-22")
FROM_DATE = INICIO_OPERACIONES - pd.Timedelta(days=365*3)
TO_DATE = INICIO_OPERACIONES + pd.Timedelta(days=365*3)

### 0. Reading data

#### Spatial Data

In [3]:
vialidades = (
    gpd
    .read_file(os.path.join(PATH_DATA, 'vialidades.json'))
    .assign(
        tipo_vialidad=lambda x: x.TIPO_VIA.map({
            "Vía primaria":"primaria", 
            'Vía de acceso controlado':'acceso_controlado'
        }),
        carriles=lambda x: pd.to_numeric(x.CARRILES, errors='raise'),
        sentidos=lambda x: x.CIRCULA.map({
            "Un sentido":1, 
            "Dos sentidos":2, 
            "Un sentido con carril de contraflujo":2
        })
    )
    .rename({
        'NIVEL':'nivel', 
        'NOMENCLAT':'calle', 
        'NOMBRE':'vialidad', 
        'ID_VIA':'id_via'
    }, axis=1)
    [[
        'id_via', 'vialidad', 'calle', 'tipo_vialidad', 
        'carriles', 'sentidos', 'nivel', 'geometry'
    ]]
)

speed_cameras = gpd.read_file(
    os.path.join(
        PATH_DATA,
        'fotocivicas-ubicacion-puntos',
        'fotocivicas-ubicacion-puntos.shp'
    )
)

#### Temporal Data

In [4]:
coordinates = (
    pd
    .read_parquet(os.path.join(PATH_DATA, "metro-station-coordinates.parquet"))
    .pipe(lambda df: (
        gpd
        .GeoDataFrame(
            data=df.drop(['latitude', 'longitude'], axis=1),
            geometry=gpd.points_from_xy(df.longitude, df.latitude),
            crs="EPSG:4326"
        )
    ))
    .set_geometry('geometry')
)

afluencia = (
    pd
    .read_parquet(os.path.join(PATH_DATA, "afluencia-metro-semanal.parquet"))
    .merge(coordinates, on='estacion')
)

accidents = (
    pd
    .read_parquet(os.path.join(PATH_DATA, "classified-incidents.parquet"))
    .pipe(lambda df: (
        gpd
        .GeoDataFrame(
            data=df.drop(['latitude', 'longitude'], axis=1),
            geometry=gpd.points_from_xy(df.longitude, df.latitude),
            crs="EPSG:4326"
        )
    ))
)

volumen_mensual = (
    pd.read_parquet(os.path.join(PATH_DATA, "volumen-total-mensual.parquet"))
)

# Hay que construir el volumen diario del tráfico 
# NOTA IMPORTANTE: SE ASUME QUE EL FLUJO DE TRÁFICO DIARIO FUE LINEAL CON BASE EN EL VOLUMEN MENSUAL TOTAL
# BÁSICAMENTE, SOLO ESTOY DIVIDIENDO EL FLUJO MENSUAL ENTRE EL NÚMERO DE DÍAS PARA LUEGO PODER CALCULAR
# EL PROMEDIO DE TRÁFICO SEMANAL
volumen_mensual["end"] = volumen_mensual.timestamp + pd.offsets.MonthEnd(1)
daily_list = []

for _, row in volumen_mensual.iterrows():
    start = row['timestamp']
    end = row['end']
    days = (end - start).days + 1

    # volumen diario lineal
    daily_value = row['volumen_mensual'] / days

    # crear rango diario
    r = pd.date_range(start, end, freq='D')

    # dataframe diario para el mes
    temp = pd.DataFrame({
        'fecha': r,
        'volumen_diario': daily_value
    })
    
    daily_list.append(temp)
    
volumen_semanal = (
    pd
    .concat(daily_list, ignore_index=True)
    .groupby(pd.Grouper(key='fecha', freq="1W"))
    .agg(
        volumen_semanal=pd.NamedAgg("volumen_diario", "sum")
    )
    .reset_index()
)

### 1. Building the grid
Esto ya lo había hecho, pero lo volveré a hacer para acordarme qué chingados fue lo que hice.

In [5]:
def _build_grid(df:gpd.GeoDataFrame, n:int) -> gpd.GeoDataFrame:
    '''
    Esta función recibe un dataframe con la información 
    de geometría de líneas y regresa una cuadrícula 
    del tamaño n especificado, pero solo aquellas 
    que tengan overlap con las líneas dadas.
    
    :param df: GeoDataFrame con LineStrings
    :param n: total de cuadrados a hacer
    
    :return: gpd.GeoDataFrame con dos columnas grid_id y geometry 
    '''
    def create_grid(upper_left, lower_right, n):
        minx, maxy = upper_left
        maxx, miny = lower_right
    
        # Calcular tamaño de cada celda
        cell_width = (maxx - minx) / n
        cell_height = (maxy - miny) / n
    
        # Crear una lista para guardar los polígonos
        polygons = []
        
        # Crear cuadrícula
        for i in range(n):
            for j in range(n):
                x1 = minx + i * cell_width
                y1 = maxy - j * cell_height
                x2 = x1 + cell_width
                y2 = y1 - cell_height
                polygons.append(Polygon([(x1, y1), (x2, y1), (x2, y2), (x1, y2)]))
        
        # Crear un GeoDataFrame a partir de los polígonos
        result = (
            gpd
            .GeoDataFrame({'geometry': polygons}, crs="EPSG:4326")
            .reset_index()
            .rename({'index':'grid_id'}, axis=1)
            # para calcular la distancia a la estación del metro más cercana
            .assign(centroid=lambda x: x.centroid) 
        )
    
        return result
        
    minx, miny, maxx, maxy = df.total_bounds

    upper_left = (minx, maxy)
    lower_right = (maxx, miny)
    
    grid = create_grid(upper_left, lower_right, n)
    
    # Creamos el match para quitar los del grid que no tengan ninguno
    squares_with_streets = (
        gpd
        .sjoin(df, grid, how="left", predicate="intersects")
        [['grid_id']]
        .drop_duplicates()
    )
    
    final_grid = grid.merge(squares_with_streets, on='grid_id')

    # Calculamos el length en metros (para referencia)
    import math
    area = final_grid.to_crs(epsg=6933).area.values[0]
    length = round(math.sqrt(area), 2)

    return {
        "grid":final_grid,
        "length_m":length
    }

In [6]:
n = GRID_SIZE
grid_response = build_grid(vialidades, n)
grid = grid_response.get("grid")
length = grid_response.get("length_m")

### 2. Assigning features
- Treatment: grid has camera or not
- Fixed features:
    - Número de vialidades
    - Número máximo de carriles
    - Si hay o no dos direcciones
    - Número de vialidades primarias
    - Número de vialidades de acceso controlado
    - Nivel máximo
- Afluencia del metro
    - Media semanal
    - STD semanal
    - Distancia a la estación más cercana
- Tendencia de accidentes
    - Agrupado mensual por nivel del incidente y por totales
    - Media
    - Desviación estándar

In [7]:
# I. Speed Cameras
grid_has_camera = (
    gpd
    .sjoin(
        left_df=grid,
        right_df=speed_cameras,
        how="left", 
        predicate="intersects"
    )
    .assign(has_camera=lambda x: ~x.index_right.isna())
    .astype({'has_camera':'int'})
    [['grid_id', 'has_camera']]
    .drop_duplicates(keep='last', ignore_index=True)
)

In [8]:
# II. Fixed features de las vialidades
# n_vialidades, max_carriles, both_directions, via_primaria, via_acc_cont, max_nivel
fixed_features = (
    gpd
    .sjoin(
        left_df=grid,
        right_df=vialidades,
        how="left", 
        predicate="intersects"
    )
    .groupby("grid_id")
    .agg(
        n_vialidades=pd.NamedAgg("id_via", "nunique"),
        max_carriles=pd.NamedAgg("carriles", "max"),
        both_directions=pd.NamedAgg("sentidos", lambda x: int(2 in list(x))),
        via_primaria=pd.NamedAgg("tipo_vialidad", lambda x: int("primaria" in set(x))),
        via_acc_cont=pd.NamedAgg("tipo_vialidad", lambda x: int("acceso_controlado" in set(x))),
        max_nivel=pd.NamedAgg("nivel", "max")
    )
    .reset_index()
)

In [9]:
# III. Afluencia en el metro
affluence = (
    # A ver, quiero poder incorporar de cierta manera el volumen de tráfico en la ciudad 
    # y la afluencia en las estaciones del metro, para lo cual voy a calcular primero el
    # total de afluencia semanal en cada estación, luego le voy a pegar el volumen semanal
    # de tráfico en la ciudad, y lo voy a divir, con eso voy a tener una tasa de afluencia 
    # semanal por cada 1_000 vehículos, así ya puedo comparar un poco mejor
    afluencia
    .query('before_treatment')
    .groupby([
        'estacion', 
        'geometry', 
        pd.Grouper(key='fecha', freq='1M')
    ])
    .agg(afluencia_mensual=pd.NamedAgg('afluencia_total', 'sum'))
    .reset_index()
    .merge(volumen_mensual, left_on='fecha', right_on='end')
    .assign(tasa_afluencia_mensual=lambda x: x.afluencia_mensual / x.volumen_mensual)
    # Ahora sí podemos calcular los valores sobre las tasas
    # La ventaja de esto es que encapsula la posible variabilidad 
    # que cada estación tenga con respecto al volumen de tráfico
    .groupby(["estacion", "geometry"])
    .agg(
        # media de afluencia semanal (en promedio, cuánta gente usaba cada estación a la semana)
        mean_afluencia_mensual=pd.NamedAgg("tasa_afluencia_mensual", "mean"), 
        # desviación estándar de la afluencia semana en cada estacion
        std_afluencia_mensual=pd.NamedAgg("tasa_afluencia_mensual", "std"), 
    )
    .reset_index()
    # Una vez que tenemos los datos de las estaciones, hay que poder calcular 
    # la distancia entre cada celda y la más cercana
    .pipe(lambda df: (
        gpd
        .sjoin_nearest(
            # transformamos ambos sistemas de referencia a metros
            left_df=grid.set_geometry('centroid').to_crs(epsg=6933), 
            right_df=df.set_geometry('geometry').to_crs(epsg=6933),
            how='inner', 
            distance_col='distance_to_station'
        )
    ))
    .drop_duplicates(subset='grid_id')
    [["grid_id", "mean_afluencia_mensual", "std_afluencia_mensual", "distance_to_station"]]
)

In [10]:
# IV. Accidentes
# Quiero calcular cómo se mueven las tasas de accidentes para cada mes 
# (una semana es muy poco, no me da mucha información) en un grid diario
# Para lo cual voy a calcular el promedio de accidentes mensuales

accidents_monthly = (
    gpd
    # Le pegamos a cada accidente el grid en el que ocurrieron
    .sjoin(
        left_df=grid,
        right_df=accidents.query('before_treatment'),
        how='inner', 
        predicate='intersects'
    )
    # Hacemos un conteo de accidentes semanal para cada celda por nivel
    .assign(
        min=lambda x: (x.incident_level == 'MIN').astype(int),
        pic=lambda x: (x.incident_level == 'PIC').astype(int),
        fcs=lambda x: (x.incident_level == 'FCS').astype(int)
    )
    .groupby([
        pd.Grouper(key='grid_id'),
        pd.Grouper(key='incident_level'),
        pd.Grouper(key='timestamp', freq='1M')
    ])
    .agg(
        total=pd.NamedAgg('folio', 'count'),
        min=pd.NamedAgg('min', 'sum'),
        pic=pd.NamedAgg('pic', 'sum'),
        fcs=pd.NamedAgg('fcs', 'sum')
    )
    .reset_index()
)

full_months = pd.period_range('2016-04-01', '2022-04-01', freq='M')
unique_grids = accidents_monthly['grid_id'].unique()
unique_levels = accidents_monthly['incident_level'].unique()

full_index = pd.MultiIndex.from_product(
    [unique_grids, unique_levels, full_months],
    names=['grid_id', 'incident_level', 'timestamp']
)

monthly_accidents_tendencies = (
    accidents_monthly
    .assign(timestamp=lambda x: x.timestamp.dt.to_period("M"))
    .set_index(['grid_id', 'incident_level', 'timestamp'])
    .reindex(full_index)
    .reset_index()
    .assign(
        timestamp=lambda x: x.timestamp.dt.to_timestamp()
    )
    .drop(columns=['timestamp', 'incident_level'])
    .groupby(['grid_id'])
    .agg(["mean", "std"])
)

monthly_accidents_tendencies.columns = [
    f"{stat}_{metric}" for metric, stat in monthly_accidents_tendencies.columns
]
monthly_accidents_tendencies.reset_index(inplace=True)

### Building final df

In [11]:
final_ps_features = (
    grid_has_camera
    .merge(fixed_features)
    .merge(affluence)
    .merge(monthly_accidents_tendencies)
)
final_ps_features.to_parquet(
    os.path.join(PATH_DATA, "grid-propensity-score-features.parquet"), 
    index=False
)

In [12]:
grid.to_feather(os.path.join(PATH_DATA, "grid.parquet"), index=False)

### Designing the outcome variable
Vamos a construir el outcome variable para cada uno de los grids, que no es otra cosa más que la tasa de accidentes en el grid a lo largo del tiempo

In [13]:
# Hay que encontrar primero: ¿cuántos accidentes sucedieron semanalmente en cada grid?
# Luego le pegamos el total de tráfico que hubo en el mes
# Y lo dividimos
all_accidents_monthly = (
    gpd
    # Le pegamos a cada accidente el grid en el que ocurrieron
    .sjoin(
        left_df=grid,
        right_df=accidents,
        how='inner', 
        predicate='intersects'
    )
    # Hacemos un conteo de accidentes semanal para cada celda por nivel
    .assign(
        min=lambda x: (x.incident_level == 'MIN').astype(int),
        pic=lambda x: (x.incident_level == 'PIC').astype(int),
        fcs=lambda x: (x.incident_level == 'FCS').astype(int)
    )
    .groupby([
        pd.Grouper(key='grid_id'),
        pd.Grouper(key='incident_level'),
        pd.Grouper(key='timestamp', freq='1M')
    ])
    .agg(
        total=pd.NamedAgg('folio', 'count'),
        min=pd.NamedAgg('min', 'sum'),
        pic=pd.NamedAgg('pic', 'sum'),
        fcs=pd.NamedAgg('fcs', 'sum')
    )
    .reset_index()
)

full_months = pd.period_range('2016-04-01', '2022-04-01', freq='M')
unique_grids = all_accidents_monthly['grid_id'].unique()
unique_levels = all_accidents_monthly['incident_level'].unique()

full_index = pd.MultiIndex.from_product(
    [unique_grids, unique_levels, full_months],
    names=['grid_id', 'incident_level', 'timestamp']
)

outcome = (
    all_accidents_monthly
    .assign(timestamp=lambda x: x.timestamp.dt.to_period("M"))
    .set_index(['grid_id', 'incident_level', 'timestamp'])
    .reindex(full_index)
    .fillna(0)
    .reset_index()
    .assign(timestamp=lambda x: x.timestamp.dt.to_timestamp())
    .merge(volumen_mensual, on='timestamp')
    # .assign(
    #     tasa_total=lambda x: x.total/x.volumen_mensual,
    #     tasa_min=lambda x: x['min']/x.volumen_mensual,
    #     tasa_pic=lambda x: x.pic / x.volumen_mensual,
    #     tasa_fcs=lambda x: x.fcs / x.volumen_mensual
    # )
    .drop(columns=['incident_level', 'end'])
    .sort_values(["grid_id", "timestamp"], ignore_index=True)
)

outcome.to_parquet(os.path.join(PATH_DATA, "outcome-variables.parquet"), index=False)

In [14]:
outcome.drop_duplicates(subset=["grid_id", "timestamp"])

,grid_id,timestamp,total,min,pic,fcs,volumen_mensual
0,1324,2016-04-01,0,0,0,0,14922.366917
3,1324,2016-05-01,0,0,0,0,15517.130522
6,1324,2016-06-01,0,0,0,0,15016.220080
9,1324,2016-07-01,0,0,0,0,16743.696904
12,1324,2016-08-01,0,0,0,0,16521.534794
...,...,...,...,...,...,...,...
3091827,249812,2021-12-01,1,1,0,0,16799.182227
3091830,249812,2022-01-01,0,0,0,0,14332.837840
3091833,249812,2022-02-01,0,0,0,0,12186.614917
3091836,249812,2022-03-01,0,0,0,0,15434.669592


In [15]:
grid

,grid_id,geometry,centroid
0,323,"POLYGON ((-99.30068 19.32239, -99.29999 19.322...",POINT (-99.30034 19.32205)
1,324,"POLYGON ((-99.30069 19.32171, -99.30001 19.321...",POINT (-99.30036 19.32136)
2,823,"POLYGON ((-99.29999 19.32238, -99.29931 19.322...",POINT (-99.29966 19.32203)
3,824,"POLYGON ((-99.30001 19.32170, -99.29932 19.321...",POINT (-99.29967 19.32135)
4,1324,"POLYGON ((-99.29932 19.32169, -99.29863 19.321...",POINT (-99.29898 19.32134)
...,...,...,...
19647,248816,"POLYGON ((-98.95996 19.32070, -98.95927 19.320...",POINT (-98.95962 19.32035)
19648,249313,"POLYGON ((-98.95923 19.32274, -98.95854 19.322...",POINT (-98.95889 19.32239)
19649,249314,"POLYGON ((-98.95924 19.32205, -98.95856 19.322...",POINT (-98.95891 19.32170)
19650,249812,"POLYGON ((-98.95853 19.32340, -98.95784 19.323...",POINT (-98.95819 19.32306)
